# 03 - CNN (deep learning)
A convolutional network: small filters slide over the image to find strokes and curves.

## 1. Imports and settings

In [20]:
import copy
import json
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from torchvision import datasets

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
torch.manual_seed(SEED)

## 2. Load the data
Download MNIST (first run only) and scale the pixels to 0-1.

In [21]:
train = datasets.MNIST(ROOT / "data", train=True, download=True)
test = datasets.MNIST(ROOT / "data", train=False, download=True)
x_all, y_all = train.data.numpy().astype(np.float32) / 255, train.targets.numpy()
x_test, y_test = test.data.numpy().astype(np.float32) / 255, test.targets.numpy()

## 3. Split the data
54,000 images for training and 6,000 for validation (to compare settings). The 10,000 test images are kept for the end.

In [22]:
x_train, x_val, y_train, y_val = train_test_split(x_all, y_all, test_size=6000, stratify=y_all, random_state=SEED)

## 4. Add a channel
The CNN reads images as (1, 28, 28): one grey channel, then height and width. Convert everything to PyTorch tensors.

In [23]:
x_train, x_val, x_test = (torch.from_numpy(a[:, None]) for a in (x_train, x_val, x_test))
y_train, y_val, y_test = (torch.from_numpy(a) for a in (y_train, y_val, y_test))

## 5. The model
Two blocks of Conv, ReLU, MaxPool find shapes and shrink the image. Then a small classifier gives the 10 scores.

In [24]:
class SimpleCNN(nn.Module):
    def __init__(self, filters, fc_units, dropout):
        super().__init__()
        f1, f2 = filters
        self.features = nn.Sequential(
            nn.Conv2d(1, f1, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(f1, f2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(f2 * 7 * 7, fc_units), nn.ReLU(), nn.Dropout(dropout), nn.Linear(fc_units, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

## Helper: accuracy
The share of images the model gets right.

In [25]:
def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        correct = sum((model(x[i:i + 2000]).argmax(1) == y[i:i + 2000]).sum().item() for i in range(0, len(x), 2000))
    return correct / len(x)

## Helper: train
Trains with Adam, checks the validation accuracy after every epoch, and keeps the weights of the best epoch.

In [26]:
def train(model, lr, epochs, batch_size=128):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best_acc, best_state = 0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x_train))
        for i in range(0, len(x_train), batch_size):
            batch = order[i:i + batch_size]
            opt.zero_grad()
            loss_fn(model(x_train[batch]), y_train[batch]).backward()
            opt.step()
        val_acc = accuracy(model, x_val, y_val)
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return best_acc

## 6. Find good settings
Try 3 filter sizes and 2 learning rates (3 epochs each) and keep the best one on the validation data.

In [27]:
results = []
for filters in [(8, 16), (16, 32), (32, 64)]:
    for lr in [1e-3, 3e-3]:
        val_acc = train(SimpleCNN(filters, fc_units=64, dropout=0.0), lr, epochs=3)
        results.append((val_acc, filters, lr))
        print(f"filters={filters}  learning rate={lr}  validation accuracy {val_acc:.4f}")
best_acc, best_filters, best_lr = max(results)
print("best:", best_filters, best_lr)

filters=(8, 16)  learning rate=0.001  validation accuracy 0.9745
filters=(8, 16)  learning rate=0.003  validation accuracy 0.9843
filters=(16, 32)  learning rate=0.001  validation accuracy 0.9760
filters=(16, 32)  learning rate=0.003  validation accuracy 0.9850
filters=(32, 64)  learning rate=0.001  validation accuracy 0.9813
filters=(32, 64)  learning rate=0.003  validation accuracy 0.9857
best: (32, 64) 0.003


## 7. Train the final model
Same idea with the best settings, for more epochs.

In [28]:
model = SimpleCNN(best_filters, fc_units=64, dropout=0.0)
start = time.time()
train(model, best_lr, epochs=10)
train_time = time.time() - start
print(f"training time: {train_time:.0f} s")

training time: 85 s


## 8. Check the accuracy
Test accuracy is the main result. Train accuracy shows how much the model overfits.

In [29]:
test_acc = accuracy(model, x_test, y_test)
train_acc = accuracy(model, x_train[:10000], y_train[:10000])
print(f"test accuracy {test_acc:.4f}   train accuracy {train_acc:.4f}")

test accuracy 0.9902   train accuracy 0.9955


## 9. Save the model
The web app loads this file, so it also needs the filter sizes to rebuild the network.

In [30]:
torch.save({"arch": "cnn", "config": dict(filters=list(best_filters), fc_units=64, dropout=0.0), "state_dict": model.state_dict()},
           ROOT / "app" / "models" / "cnn_model.pt")

## 10. Save the results
The comparison notebook reads this file.

In [31]:
results = dict(test_acc=test_acc, train_acc=train_acc, train_time=train_time,
               parameters=sum(p.numel() for p in model.parameters()),
               settings=dict(filters=list(best_filters), learning_rate=best_lr, dropout=0.0))
(ROOT / "artifacts" / "cnn_results.json").write_text(json.dumps(results, indent=2))

217